<a href="https://colab.research.google.com/github/PepRov/alphafold-ios/blob/main/Copy_of_AlphaFold2_v1_6_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
#@title Load Sequence from PepRop

import urllib.request
import os
import re
import hashlib

# ------------------ SETTINGS ------------------

jobname = "PepRop"
num_relax = 0

use_amber = False
use_templates = False
custom_template_path = None

# ------------------ LOAD FASTA ------------------

sequence_url = "https://alphafold-ios.vercel.app/api/fasta"

with urllib.request.urlopen(sequence_url) as response:
    fasta_text = response.read().decode("utf-8")

# Extract sequence
lines = fasta_text.splitlines()

query_sequence = "".join(
    line.strip()
    for line in lines
    if not line.startswith(">")
)

if not query_sequence:
    raise Exception("No sequence found.")

print("Sequence loaded successfully")
print(query_sequence)

# ------------------ CREATE UNIQUE JOBNAME ------------------

job_hash = hashlib.sha1(query_sequence.encode()).hexdigest()[:5]
jobname = f"{jobname}_{job_hash}"

# ------------------ CREATE OUTPUT FOLDER ------------------

os.makedirs(jobname, exist_ok=True)

# ------------------ SAVE QUERY CSV ------------------

queries_path = os.path.join(jobname, f"{jobname}.csv")

with open(queries_path, "w") as f:
    f.write(f"id,sequence\n{jobname},{query_sequence}")

# ------------------ SIMPLE SINGLE-SEQUENCE MSA ------------------

msa_mode = "single_sequence"
pair_mode = "unpaired"

a3m_file = os.path.join(jobname, f"{jobname}.single_sequence.a3m")

with open(a3m_file, "w") as f:
    f.write(">1\n%s" % query_sequence)

queries_path = a3m_file

# ------------------ SUMMARY ------------------

print("Job:", jobname)
print("Length:", len(query_sequence))

Sequence loaded successfully
EQQTYFWAG
Job: PepRop_be602
Length: 9


In [20]:
#@title Install dependencies
%%time
import os
USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  # hack to fix TF crash
  os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so /usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so")
  os.system("touch COLABFOLD_READY")

if USE_AMBER or USE_TEMPLATES:
  if not os.path.isfile("CONDA_READY"):
    print("installing conda...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if USE_TEMPLATES and not os.path.isfile("HH_READY") and USE_AMBER and not os.path.isfile("AMBER_READY"):
  print("installing hhsuite and amber...")
  os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
  os.system("touch HH_READY")
  os.system("touch AMBER_READY")
else:
  if USE_TEMPLATES and not os.path.isfile("HH_READY"):
    print("installing hhsuite...")
    os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python='{PYTHON_VERSION}'")
    os.system("touch HH_READY")
  if USE_AMBER and not os.path.isfile("AMBER_READY"):
    print("installing amber...")
    os.system(f"mamba install -y -c conda-forge openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
    os.system("touch AMBER_READY")

    # ------------------ MSA SETTINGS ------------------

msa_mode = "single_sequence"
pair_mode = "unpaired"

# create simple single-sequence A3M
a3m_file = os.path.join(jobname, f"{jobname}.single_sequence.a3m")

with open(a3m_file, "w") as text_file:
    text_file.write(">1\n%s" % query_sequence)

queries_path = a3m_file

# ------------------ MODEL SETTINGS ------------------

# Simple monomer prediction settings
model_type = "alphafold2"
msa_mode = "single_sequence"
pair_mode = "unpaired"

# Prediction quality / speed
num_recycles = 3
recycle_early_stop_tolerance = 0.5

# Relaxation
relax_max_iterations = 0
num_relax = 0

# Sampling
num_seeds = 1
use_dropout = False

# MSA
max_msa = "16:32"

# Advanced options
pairing_strategy = "greedy"
calc_extra_ptm = False

# Save options
save_all = False
save_recycles = False
save_to_google_drive = False

# Plot quality
dpi = 200

CPU times: user 233 µs, sys: 0 ns, total: 233 µs
Wall time: 272 µs


In [15]:
#@title Run Prediction

import os
import sys
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

from pathlib import Path
from colabfold.download import download_alphafold_params
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run

# Setup output folder
result_dir = jobname
log_filename = os.path.join(jobname, "log.txt")

setup_logging(Path(log_filename))

# Load query
queries, is_complex = get_queries(queries_path)

# Download AlphaFold weights
download_alphafold_params(model_type, Path("."))

# Run prediction
results = run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=custom_template_path,
    num_relax=num_relax,
    msa_mode=msa_mode,
    model_type=model_type,
    num_models=5,
    num_recycles=num_recycles,
    relax_max_iterations=relax_max_iterations,
    recycle_early_stop_tolerance=recycle_early_stop_tolerance,
    num_seeds=num_seeds,
    use_dropout=use_dropout,
    model_order=[1,2,3,4,5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=pair_mode,
    stop_at_score=100,
    dpi=dpi,
    zip_results=False,
    save_all=save_all,
    max_msa=max_msa,
    save_recycles=save_recycles,
    user_agent="colabfold/google-colab-main",
)

# Zip results
results_zip = f"{jobname}.result.zip"
os.system(f"zip -r {results_zip} {jobname}")

#@title Display 3D Structure

import py3Dmol
import glob

# Find generated PDB files
pdb_file = glob.glob(f"{jobname}/*_unrelaxed_*.pdb")

if len(pdb_file) == 0:
    raise Exception("No PDB structure file was generated.")

# Display structure
view = py3Dmol.view(width=800, height=600)
view.addModel(open(pdb_file[0], 'r').read(), 'pdb')

# Simple cartoon rendering
view.setStyle({'cartoon': {'color': 'spectrum'}})

view.zoomTo()
view.show()

2026-05-21 16:04:11,340 Running on GPU
2026-05-21 16:04:11,344 Found 2 citations for tools or databases
2026-05-21 16:04:11,344 Query 1/1: PepRop_be602.single_sequence (length 9)
2026-05-21 16:04:32,743 alphafold2_model_1_seed_000 recycle=0 pLDDT=78.9
2026-05-21 16:04:46,723 alphafold2_model_1_seed_000 recycle=1 pLDDT=80.1 tol=0.247
2026-05-21 16:04:46,724 alphafold2_model_1_seed_000 took 28.4s (1 recycles)
2026-05-21 16:04:46,837 alphafold2_model_2_seed_000 recycle=0 pLDDT=71.8
2026-05-21 16:04:46,938 alphafold2_model_2_seed_000 recycle=1 pLDDT=71.9 tol=0.438
2026-05-21 16:04:46,939 alphafold2_model_2_seed_000 took 0.2s (1 recycles)
2026-05-21 16:04:47,037 alphafold2_model_3_seed_000 recycle=0 pLDDT=80.9
2026-05-21 16:04:47,126 alphafold2_model_3_seed_000 recycle=1 pLDDT=80.6 tol=0.312
2026-05-21 16:04:47,126 alphafold2_model_3_seed_000 took 0.2s (1 recycles)
2026-05-21 16:04:47,216 alphafold2_model_4_seed_000 recycle=0 pLDDT=80.2
2026-05-21 16:04:47,305 alphafold2_model_4_seed_000 re

3Dmol.js failed to load for some reason. Please check your browser console for error messages.